<a href="https://colab.research.google.com/github/worasak2530seed/fast-hybrid-pro/blob/main/notebooks/Early_Signal_Scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fast Hybrid Pro — EARLY ENTRY / EARLY WARNING Scanner

**Standalone scanner. After-hours mode uses the latest verified completed daily OHLCV bar; it is not a live intraday signal. This is not the Production notebook.**

- Loads SET/mai listings from the public Thai Securities Data GitHub JSON dataset, validates its update timestamp, schema and market row counts, and falls back to the official SET listing if the mirror is stale/unavailable. This dataset supplies listings/classification, not intraday prices.
- Uses Yahoo Finance (`yfinance`) for OHLCV and current market-cap lookup; no SETSMART or SMART Marketplace.
- Applies the agreed universe filters before downloading the longer indicator history.
- Uses daily OHLCV for the agreed universe filters and completed 15-minute OHLCV for current entry/warning signals during the SET session. It does **not** run a backtest, parameter sweep, place orders, write reports to `main`, or trigger GitHub Actions.
- The 20-day trading value is estimated as `Close × Volume` because Yahoo OHLCV does not provide the exchange's exact daily traded-value field in this workflow. Review this limitation before relying on borderline cases.

Run the cells from top to bottom. The first scan may take several minutes due to Yahoo Finance requests.

In [ ]:
# Install runtime dependencies and check out the isolated scanner branch.
%pip -q install yfinance pandas numpy requests openpyxl xlrd

import io
import re
import sys
import time
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from IPython.display import display

REPO_URL = "https://github.com/worasak2530seed/fast-hybrid-pro.git"
BRANCH = "main"
REPO_DIR = Path("/content/fast-hybrid-pro-scanner-source")

if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)

# Force-load the exact checked-out file. This avoids stale package/module caches
# and avoids Python resolving a different src package earlier on sys.path.
import importlib
import importlib.util
importlib.invalidate_caches()
if str(REPO_DIR) in sys.path:
    sys.path.remove(str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR))

CHECKED_OUT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
SCANNER_SOURCE_PATH = (REPO_DIR / "src" / "early_signal_scanner.py").resolve()
SCANNER_SOURCE_TEXT = SCANNER_SOURCE_PATH.read_text(encoding="utf-8")
if "def scan_intraday_signal(" not in SCANNER_SOURCE_TEXT:
    raise ImportError(
        "CHECKOUT_MISSING_SCAN_INTRADAY_SIGNAL: checked-out commit "
        + CHECKED_OUT_COMMIT + " at " + str(SCANNER_SOURCE_PATH)
    )

# Make sure the parent package points only to this checkout, then load by file path.
import src
src.__path__ = [str((REPO_DIR / "src").resolve())]
sys.modules.pop("src.early_signal_scanner", None)
_spec = importlib.util.spec_from_file_location("src.early_signal_scanner", SCANNER_SOURCE_PATH)
if _spec is None or _spec.loader is None:
    raise ImportError("SCANNER_MODULE_SPEC_FAILED: " + str(SCANNER_SOURCE_PATH))
_scanner_module = importlib.util.module_from_spec(_spec)
sys.modules["src.early_signal_scanner"] = _scanner_module
_spec.loader.exec_module(_scanner_module)

CONFIG = _scanner_module.CONFIG
scan_universe = _scanner_module.scan_universe
scan_intraday_signal = _scanner_module.scan_intraday_signal
assert callable(scan_intraday_signal), "INTRADAY_SIGNAL_IMPORT_FAILED"

print("SCANNER_BRANCH_CHECKOUT_PASS")
print("SCANNER_COMMIT:", CHECKED_OUT_COMMIT)
print("SCANNER_SOURCE_PATH:", SCANNER_SOURCE_PATH)
print("INTRADAY_SIGNAL_IMPORT_PASS:", callable(scan_intraday_signal))
print("CONFIG:", CONFIG)


In [ ]:
exec(compile("# Current market-data scan — no backtest or order execution.\n\nUNIVERSE_MARKETS = {\"SET\", \"MAI\"}\nEXCLUDED_SECTORS = {\"BANK\", \"FIN\", \"INSUR\", \"PF&REIT\", \"PROP\", \"TOURISM\"}\nMIN_MARKET_CAP_MBAHT = 5000.0\nMIN_TOTAL20_VALUE_MBAHT = 20.0\nMIN_PRICE_BAHT = 1.0\nMAX_PRICE_BAHT = 80.0\nCLASSIFICATION_SOURCE_USED = None\nCLASSIFICATION_SOURCE_UPDATED_AT = None\nHEADERS = {\"User-Agent\": \"Mozilla/5.0 (compatible; Fast-Hybrid-Pro-Scanner/1.0)\", \"Accept\": \"*/*\"}\n\ndef _flatten_columns(frame):\n    frame = frame.copy()\n    if isinstance(frame.columns, pd.MultiIndex):\n        frame.columns = [\" \".join(str(x).strip() for x in col if str(x).strip() and str(x) != \"nan\") for col in frame.columns]\n    else:\n        frame.columns = [str(c).strip() for c in frame.columns]\n    return frame\n\ndef _find_col(frame, names):\n    lookup = {str(c).strip().lower(): c for c in frame.columns}\n    for name in names:\n        if name.lower() in lookup:\n            return lookup[name.lower()]\n    return None\n\ndef _normalize_classification(raw):\n    raw = _flatten_columns(raw)\n    symbol_col = _find_col(raw, [\"symbol\", \"ticker\"])\n    market_col = _find_col(raw, [\"market\", \"market segment\"])\n    sector_col = _find_col(raw, [\"sector\", \"sector name\"])\n    company_col = _find_col(raw, [\"company name\", \"company\", \"name\", \"security name\"])\n    industry_col = _find_col(raw, [\"industry group\", \"industry\"])\n    if not all([symbol_col, market_col, sector_col]):\n        raise ValueError(f\"Classification missing required fields; columns={list(raw.columns)}\")\n    out = pd.DataFrame({\n        \"Symbol\": raw[symbol_col].astype(str).str.strip().str.upper(),\n        \"Market\": raw[market_col].astype(str).str.strip().str.upper(),\n        \"Sector\": raw[sector_col].astype(str).str.strip().str.upper(),\n        \"Industry\": raw[industry_col].astype(str).str.strip().str.upper() if industry_col else \"\",\n        \"Company\": raw[company_col].astype(str).str.strip() if company_col else \"\",\n    })\n    if not company_col:\n        raise ValueError(\"Classification has no company-name field; cannot safely enforce ordinary-share-only gate\")\n    ordinary = out[\"Company\"].str.contains(r\"PUBLIC COMPANY LIMITED|PUBLIC CO\\.?\\s*,?\\s*LTD\\.?|PCL\\.?$\", case=False, regex=True, na=False)\n    out = out[ordinary].copy()\n    sector_map = {\n        \"BANKING\": \"BANK\", \"FINANCE & SECURITIES\": \"FIN\", \"INSURANCE\": \"INSUR\",\n        \"PROPERTY DEVELOPMENT\": \"PROP\", \"PROPERTY FUND & REITS\": \"PF&REIT\",\n        \"TOURISM & LEISURE\": \"TOURISM\",\n    }\n    out[\"Sector\"] = out[\"Sector\"].replace(sector_map)\n    out = out[out[\"Market\"].isin(UNIVERSE_MARKETS)]\n    out = out[out[\"Symbol\"].ne(\"\") & out[\"Symbol\"].ne(\"NAN\")].drop_duplicates(\"Symbol\")\n    if out.empty:\n        raise ValueError(\"No SET/mai listed companies remained after classification normalization\")\n    return out.reset_index(drop=True)\n\ndef load_set_classification():\n    global CLASSIFICATION_SOURCE_USED, CLASSIFICATION_SOURCE_UPDATED_AT\n    base_url = \"https://raw.githubusercontent.com/lumduan/thai-securities-data/main/\"\n    metadata_url = base_url + \"metadata_en.json\"\n    mirror_error = None\n\n    # Prefer the separate SET/mai files and validate their schema/counts against\n    # the dataset's own update metadata before using them as current classification.\n    try:\n        meta_response = requests.get(metadata_url, headers=HEADERS, timeout=25)\n        meta_response.raise_for_status()\n        metadata = meta_response.json()\n        updated = pd.to_datetime(metadata.get(\"last_updated\"), errors=\"coerce\")\n        if pd.isna(updated):\n            raise ValueError(\"Thai Securities Data metadata has no valid last_updated timestamp\")\n        if updated.tzinfo is not None:\n            age_hours = (pd.Timestamp.now(tz=\"UTC\") - updated.tz_convert(\"UTC\")).total_seconds() / 3600.0\n        else:\n            # The upstream metadata timestamp has no timezone suffix; allow the\n            # maximum Thailand/UTC offset ambiguity while still rejecting stale data.\n            source_now = pd.Timestamp.now(tz=\"UTC\").tz_localize(None)\n            age_hours = (source_now - updated).total_seconds() / 3600.0\n        if age_hours < -8 or age_hours > 7 * 24:\n            raise ValueError(\n                f\"Thai Securities Data metadata is stale or future-dated: \"\n                f\"last_updated={updated.isoformat()}, age_hours={age_hours:.1f}\"\n            )\n        expected_markets = metadata.get(\"markets\", {})\n        expected_total = int(metadata.get(\"total_securities\", 0))\n        if expected_total <= 0:\n            raise ValueError(\"Thai Securities Data metadata has invalid total_securities\")\n\n        try:\n            market_frames = []\n            market_urls = [\n                (\"SET\", \"thai_securities_market_set_en.json\"),\n                (\"MAI\", \"thai_securities_market_mai_en.json\"),\n            ]\n            for expected_market, filename in market_urls:\n                url = base_url + filename\n                response = requests.get(url, headers=HEADERS, timeout=25)\n                response.raise_for_status()\n                payload = response.json()\n                if not isinstance(payload, list) or not payload:\n                    raise ValueError(f\"{filename} must contain a non-empty JSON list\")\n                frame = pd.DataFrame(payload)\n                required = {\"symbol\", \"name\", \"market\", \"sector\"}\n                missing = sorted(required - set(frame.columns))\n                if missing:\n                    raise ValueError(f\"{filename} missing schema fields: {missing}\")\n                actual_markets = set(frame[\"market\"].astype(str).str.strip().str.upper())\n                if actual_markets != {expected_market}:\n                    raise ValueError(\n                        f\"{filename} market labels mismatch: {sorted(actual_markets)}\"\n                    )\n                expected_count = expected_markets.get(expected_market)\n                if expected_count is None and expected_market == \"MAI\":\n                    expected_count = expected_markets.get(\"mai\")\n                if expected_count is not None and len(frame) != int(expected_count):\n                    raise ValueError(\n                        f\"{filename} count mismatch: got {len(frame)}, expected {expected_count}\"\n                    )\n                market_frames.append(frame)\n            raw = pd.concat(market_frames, ignore_index=True)\n            if len(raw) != expected_total:\n                raise ValueError(\n                    f\"SET+mai row count {len(raw)} does not match metadata total {expected_total}\"\n                )\n            result = _normalize_classification(raw)\n            if result.empty:\n                raise ValueError(\"Thai Securities Data normalized to an empty listing\")\n            CLASSIFICATION_SOURCE_USED = (\n                base_url + \"thai_securities_market_set_en.json; \"\n                + base_url + \"thai_securities_market_mai_en.json\"\n            )\n            CLASSIFICATION_SOURCE_UPDATED_AT = str(metadata.get(\"last_updated\"))\n            print(\n                \"Thai Securities Data validated:\",\n                f\"{len(raw)} raw rows; metadata age {age_hours:.1f} hours\"\n            )\n            return result\n        except Exception as split_exc:\n            # The combined file is a guarded fallback, not an unchecked alternative.\n            all_url = base_url + \"thai_securities_all_en.json\"\n            response = requests.get(all_url, headers=HEADERS, timeout=25)\n            response.raise_for_status()\n            payload = response.json()\n            raw = pd.DataFrame(payload if isinstance(payload, list) else list(payload.values()))\n            if len(raw) != expected_total:\n                raise ValueError(\n                    f\"Combined JSON row count {len(raw)} does not match metadata total {expected_total}\"\n                ) from split_exc\n            result = _normalize_classification(raw)\n            if result.empty:\n                raise ValueError(\"Thai Securities Data combined listing normalized to empty\")\n            CLASSIFICATION_SOURCE_USED = all_url\n            CLASSIFICATION_SOURCE_UPDATED_AT = updated.isoformat()\n            print(\n                \"Thai Securities Data split files unavailable; validated combined JSON:\",\n                type(split_exc).__name__\n            )\n            return result\n    except Exception as exc:\n        mirror_error = f\"{type(exc).__name__}: {exc}\"\n        print(\"Thai Securities Data mirror unavailable/untrusted; trying official SET listing:\", mirror_error)\n\n    # Official SET listing is the fallback if the public mirror is stale, malformed,\n    # or unreachable. Never silently accept an unverified mirror snapshot.\n    official_url = \"https://www.set.or.th/dat/eod/listedcompany/static/listedCompanies_en_US.xls\"\n    try:\n        response = requests.get(official_url, headers=HEADERS, timeout=30)\n        response.raise_for_status()\n        raw = None\n        try:\n            raw = pd.read_excel(io.BytesIO(response.content))\n        except Exception:\n            tables = pd.read_html(io.BytesIO(response.content))\n            if tables:\n                raw = tables[0]\n        if raw is None:\n            raise ValueError(\"Official SET listing response could not be parsed as a table\")\n        result = _normalize_classification(raw)\n        CLASSIFICATION_SOURCE_USED = official_url\n        CLASSIFICATION_SOURCE_UPDATED_AT = None\n        return result\n    except Exception as official_exc:\n        raise RuntimeError(\n            \"CLASSIFICATION_SOURCE_FAILED: Thai Securities Data mirror error=\"\n            + str(mirror_error)\n            + \"; official SET listing error=\"\n            + f\"{type(official_exc).__name__}: {official_exc}\"\n        ) from official_exc\n\ndef normalize_ohlcv(raw):\n    if raw is None or raw.empty:\n        return pd.DataFrame()\n    frame = raw.copy()\n    if isinstance(frame.columns, pd.MultiIndex):\n        levels = [set(map(str, frame.columns.get_level_values(i))) for i in range(frame.columns.nlevels)]\n        price_names = {\"Open\", \"High\", \"Low\", \"Close\", \"Volume\"}\n        chosen = next((i for i, values in enumerate(levels) if price_names.issubset(values)), 0)\n        frame.columns = frame.columns.get_level_values(chosen)\n    frame = frame.loc[:, ~frame.columns.duplicated()].copy()\n    rename = {str(c).strip().title(): str(c).strip().title() for c in frame.columns}\n    frame.columns = [rename[str(c).strip().title()] for c in frame.columns]\n    required = [\"Open\", \"High\", \"Low\", \"Close\", \"Volume\"]\n    missing = [c for c in required if c not in frame.columns]\n    if missing:\n        raise ValueError(f\"Missing OHLCV columns {missing}; got {list(frame.columns)}\")\n    frame = frame[required].copy()\n    frame.index = pd.to_datetime(frame.index)\n    if getattr(frame.index, \"tz\", None) is not None:\n        frame.index = frame.index.tz_localize(None)\n    for col in required:\n        frame[col] = pd.to_numeric(frame[col], errors=\"coerce\")\n    frame = frame.dropna(subset=[\"Open\", \"High\", \"Low\", \"Close\", \"Volume\"])\n    frame = frame[~frame.index.duplicated(keep=\"last\")].sort_index()\n    # Exclude today's possibly incomplete daily candle (Thailand local date).\n    today_bkk = pd.Timestamp.now(tz=\"Asia/Bangkok\").date()\n    frame = frame[frame.index.date < today_bkk]\n    return frame\n\ndef get_market_cap_baht(ticker):\n    try:\n        value = yf.Ticker(ticker).fast_info.get(\"market_cap\")\n        if value is not None and np.isfinite(float(value)):\n            return float(value)\n    except Exception:\n        pass\n    try:\n        value = yf.Ticker(ticker).info.get(\"marketCap\")\n        if value is not None and np.isfinite(float(value)):\n            return float(value)\n    except Exception:\n        pass\n    return np.nan\n\ndef extract_ticker_frame(batch_frame, ticker, batch_size):\n    if batch_frame is None or batch_frame.empty:\n        return pd.DataFrame()\n    if isinstance(batch_frame.columns, pd.MultiIndex):\n        for level in range(batch_frame.columns.nlevels):\n            if ticker in set(map(str, batch_frame.columns.get_level_values(level))):\n                try:\n                    item = batch_frame.xs(ticker, axis=1, level=level, drop_level=True)\n                    return normalize_ohlcv(item)\n                except Exception:\n                    continue\n        if batch_size == 1:\n            return normalize_ohlcv(batch_frame)\n        return pd.DataFrame()\n    return normalize_ohlcv(batch_frame) if batch_size == 1 else pd.DataFrame()\n\ndef download_yahoo_history(ticker, period, attempts=2):\n    \"\"\"Download one ticker with bounded retries; empty symbol results fail quickly.\"\"\"\n    last_error = None\n    for attempt in range(1, attempts + 1):\n        try:\n            raw = yf.download(\n                ticker, period=period, interval=\"1d\", auto_adjust=False,\n                progress=False, threads=False, timeout=20,\n            )\n            frame = normalize_ohlcv(raw)\n            if not frame.empty:\n                return frame\n            last_error = \"Yahoo returned no OHLCV rows\"\n        except Exception as exc:\n            last_error = f\"{type(exc).__name__}: {exc}\"\n        if attempt < attempts:\n            time.sleep(1.5 * attempt)\n    print(f\"YAHOO_TICKER_UNAVAILABLE {ticker} | {last_error}\")\n    return pd.DataFrame()\n\ndef get_official_set_latest_trading_date(expected_scan_date):\n    \"\"\"\n    Validate the common completed Yahoo daily-bar date against official SET pages.\n    The official SET pages may render their tables client-side, so use a guarded\n    cross-symbol Yahoo-date fallback only when the official date text is unreadable.\n    Never accept a future date, a weekend, a known official holiday, or stale data.\n    \"\"\"\n    from html import unescape\n    from datetime import timedelta, date\n\n    expected_scan_date = pd.Timestamp(expected_scan_date).date()\n    today = pd.Timestamp.now(tz=\"Asia/Bangkok\").date()\n    earliest_acceptable = today - timedelta(days=4)\n    if expected_scan_date < earliest_acceptable or expected_scan_date >= today:\n        raise RuntimeError(\n            f\"MARKET_DATE_STALE: common Yahoo bar date {expected_scan_date} is not recent \"\n            f\"(acceptable range: {earliest_acceptable} to {today - timedelta(days=1)}).\"\n        )\n    if expected_scan_date.weekday() >= 5:\n        raise RuntimeError(f\"MARKET_DATE_INVALID: common Yahoo bar date {expected_scan_date} is a weekend.\")\n\n    reference_urls = [\n        \"https://www.set.or.th/th/market/statistics/five-days\",\n        \"https://www.set.or.th/en/market/statistics/five-days\",\n        \"https://www.set.or.th/th/market/product/stock/overview?market=SET\",\n        \"https://www.set.or.th/en/market/product/stock/overview?market=SET\",\n    ]\n    thai_months = {\n        \"ม.ค.\": 1, \"มกราคม\": 1, \"ก.พ.\": 2, \"กุมภาพันธ์\": 2,\n        \"มี.ค.\": 3, \"มีนาคม\": 3, \"เม.ย.\": 4, \"เมษายน\": 4,\n        \"พ.ค.\": 5, \"พฤษภาคม\": 5, \"มิ.ย.\": 6, \"มิถุนายน\": 6,\n        \"ก.ค.\": 7, \"กรกฎาคม\": 7, \"ส.ค.\": 8, \"สิงหาคม\": 8,\n        \"ก.ย.\": 9, \"กันยายน\": 9, \"ต.ค.\": 10, \"ตุลาคม\": 10,\n        \"พ.ย.\": 11, \"พฤศจิกายน\": 11, \"ธ.ค.\": 12, \"ธันวาคม\": 12,\n    }\n    english_months = {\n        name.lower(): number for number, name in enumerate(\n            [\"\", \"January\", \"February\", \"March\", \"April\", \"May\", \"June\",\n             \"July\", \"August\", \"September\", \"October\", \"November\", \"December\"], 0\n        ) if name\n    }\n    def visible_text(html):\n        return unescape(re.sub(r\"<[^>]+>\", \" \", str(html))).replace(\"&nbsp;\", \" \")\n\n    official_holidays = set()\n    holiday_calendar_loaded = False\n    holiday_url = f\"https://www.set.or.th/en/about/event-calendar/holiday?year={today.year}\"\n    try:\n        response = requests.get(holiday_url, headers=HEADERS, timeout=20)\n        response.raise_for_status()\n        holiday_text = re.sub(r\"\\s+\", \" \", visible_text(response.text))\n        for month_name, month_num in english_months.items():\n            for hit in re.finditer(\n                rf\"\\b{re.escape(month_name)}\\s+(\\d{{1,2}})(?:st|nd|rd|th)?\\b\",\n                holiday_text, flags=re.IGNORECASE\n            ):\n                try:\n                    official_holidays.add(date(today.year, month_num, int(hit.group(1))))\n                except ValueError:\n                    pass\n        for month_name, month_num in thai_months.items():\n            for hit in re.finditer(\n                rf\"(\\d{{1,2}})\\s*{re.escape(month_name)}\\s*(\\d{{4}})?\",\n                holiday_text\n            ):\n                try:\n                    year_text = hit.group(2)\n                    year_num = (\n                        int(year_text) - 543 if year_text and int(year_text) > 2400\n                        else int(year_text) if year_text else today.year\n                    )\n                    official_holidays.add(date(year_num, month_num, int(hit.group(1))))\n                except ValueError:\n                    pass\n        holiday_calendar_loaded = bool(official_holidays)\n        print(\"Official SET holiday dates parsed:\", len(official_holidays))\n    except Exception as exc:\n        print(\"Official SET holiday calendar unavailable:\", type(exc).__name__)\n\n    # A known holiday must never be accepted as a trading day.\n    if expected_scan_date in official_holidays:\n        raise RuntimeError(\n            f\"MARKET_DATE_INVALID: common Yahoo bar date {expected_scan_date} appears \"\n            \"on the official SET holiday calendar.\"\n        )\n\n    official_dates = set()\n    pages_read = 0\n    parse_errors = []\n    for reference_url in reference_urls:\n        try:\n            response = requests.get(reference_url, headers=HEADERS, timeout=20)\n            response.raise_for_status()\n            pages_read += 1\n        except Exception as exc:\n            parse_errors.append(f\"{reference_url}: {type(exc).__name__}\")\n            continue\n        page_text = re.sub(r\"\\s+\", \" \", visible_text(response.text))\n        # ISO and English dates.\n        for pattern in (\n            r\"\\b\\d{4}-\\d{2}-\\d{2}\\b\",\n            r\"\\b\\d{1,2}\\s+[A-Za-z]{3,9}\\s+\\d{4}\\b\",\n            r\"\\b[A-Za-z]{3,9}\\s+\\d{1,2}(?:st|nd|rd|th)?,?\\s+\\d{4}\\b\",\n        ):\n            for raw_date in re.findall(pattern, page_text):\n                parsed = pd.to_datetime(raw_date, errors=\"coerce\", dayfirst=True)\n                if not pd.isna(parsed):\n                    day = parsed.date()\n                    if earliest_acceptable <= day < today:\n                        official_dates.add(day)\n        # Thai Buddhist-calendar dates, including abbreviated month names.\n        thai_pattern = (\n            r\"(\\d{1,2})\\s*(ม\\.ค\\.|มกราคม|ก\\.พ\\.|กุมภาพันธ์|มี\\.ค\\.|มีนาคม|\"\n            r\"เม\\.ย\\.|เมษายน|พ\\.ค\\.|พฤษภาคม|มิ\\.ย\\.|มิถุนายน|ก\\.ค\\.|กรกฎาคม|\"\n            r\"ส\\.ค\\.|สิงหาคม|ก\\.ย\\.|กันยายน|ต\\.ค\\.|ตุลาคม|พ\\.ย\\.|พฤศจิกายน|\"\n            r\"ธ\\.ค\\.|ธันวาคม)\\s*(\\d{4})\"\n        )\n        for day_text, month_text, year_text in re.findall(thai_pattern, page_text):\n            try:\n                year_num = int(year_text)\n                if year_num > 2400:\n                    year_num -= 543\n                day = date(year_num, thai_months[month_text], int(day_text))\n                if earliest_acceptable <= day < today:\n                    official_dates.add(day)\n            except ValueError:\n                pass\n\n    # Best case: the common Yahoo date is explicitly present on SET's own page.\n    if expected_scan_date in official_dates:\n        print(\"Official market-date validation: PASS (common Yahoo date appears on SET page)\")\n        print(\"Official SET pages checked:\", pages_read)\n        print(\"Common completed scan date:\", expected_scan_date)\n        return expected_scan_date\n\n    # SET's tables may be client-rendered and unreadable to requests. In that case,\n    # the same latest daily bar across the scanner's independently downloaded stocks\n    # is used as a guarded fallback. This does NOT bypass weekend, age, or holiday checks.\n    if not official_dates:\n        print(\"Official SET date text could not be parsed; using common Yahoo daily-bar consensus.\")\n        print(\"Official SET pages fetched:\", pages_read, \"| holiday calendar parsed:\", holiday_calendar_loaded)\n        print(\"Official page request issues:\", parse_errors[:2])\n        print(\"Market-date validation mode: GUARDED_YAHOO_CONSENSUS_FALLBACK\")\n        print(\"Common completed scan date:\", expected_scan_date)\n        return expected_scan_date\n\n    # SET pages frequently include unrelated metadata dates in server-rendered HTML,\n    # while the actual market tables are client-rendered. A non-empty list of parsed\n    # dates is therefore not sufficient evidence that the Yahoo date is wrong.\n    # Keep the guarded fallback: expected_scan_date must already have passed the\n    # recency, weekday, and known-holiday checks above, and the caller has enforced\n    # one common completed daily-bar date across all qualifying symbols.\n    print(\n        \"Official SET date text did not confirm the common Yahoo date; \"\n        \"using guarded cross-symbol consensus instead.\"\n    )\n    print(\"Parsed dates from SET HTML (diagnostic only):\", sorted(official_dates))\n    print(\"Official SET pages checked:\", pages_read)\n    print(\"Holiday calendar parsed:\", holiday_calendar_loaded)\n    print(\"Market-date validation mode: GUARDED_YAHOO_CONSENSUS_FALLBACK\")\n    print(\"Common completed scan date:\", expected_scan_date)\n    return expected_scan_date\n\nprint(\"Loading SET/mai listing and sector classification...\")\nlisted = load_set_classification()\nprint(\"Classification source:\", CLASSIFICATION_SOURCE_USED)\nprint(\"Classification dataset timestamp (as reported; timezone may be unspecified):\", CLASSIFICATION_SOURCE_UPDATED_AT or \"not provided by official source\")\nunknown_sector_count = int(listed[\"Sector\"].astype(str).str.strip().str.upper().isin([\"\", \"-\", \"NAN\", \"NONE\"]).sum())\nprint(\"Listings with unspecified sector:\", unknown_sector_count, \"(not guessed from broad industry labels)\")\nlisted = listed[~listed[\"Sector\"].isin(EXCLUDED_SECTORS)].copy()\nprint(\"Candidates after sector exclusion:\", len(listed))\n\n# Stage 1: download short OHLCV only to apply price and 20-session liquidity gates.\nsymbols = listed[\"Symbol\"].tolist()\ntickers = [f\"{symbol}.BK\" for symbol in symbols]\nshort_data = {}\nmissing_short = []\nBATCH_SIZE = 20\nfor start in range(0, len(tickers), BATCH_SIZE):\n    batch = tickers[start:start + BATCH_SIZE]\n    print(f\"Downloading universe snapshot {start + 1}-{start + len(batch)} / {len(tickers)}\")\n    try:\n        raw_batch = yf.download(batch, period=\"3mo\", interval=\"1d\", auto_adjust=False, progress=False, threads=True, group_by=\"ticker\")\n    except Exception as exc:\n        print(\"Batch download failed:\", type(exc).__name__)\n        raw_batch = pd.DataFrame()\n    for ticker in batch:\n        item = pd.DataFrame()\n        try:\n            item = extract_ticker_frame(raw_batch, ticker, len(batch))\n        except Exception as exc:\n            print(f\"YAHOO_BATCH_PARSE_WARNING {ticker} | {type(exc).__name__}\")\n        # Yahoo batch responses can omit individual Thai tickers even when the\n        # rest of the batch succeeds. Retry only the missing ticker separately.\n        if len(item) < 20:\n            item = download_yahoo_history(ticker, period=\"3mo\", attempts=2)\n        if len(item) >= 20:\n            short_data[ticker] = item\n        else:\n            missing_short.append(ticker)\n\nprint(f\"Short OHLCV ready: {len(short_data)} / {len(tickers)}; unavailable/short: {len(missing_short)}\")\nstage_rows = []\nmeta_by_ticker = {}\nstale_short_data = {}\nMAX_STALE_DAILY_BAR_DAYS = 7\ntoday_bkk = pd.Timestamp.now(tz=\"Asia/Bangkok\").date()\nfor _, item in listed.iterrows():\n    ticker = f\"{item['Symbol']}.BK\"\n    frame = short_data.get(ticker)\n    if frame is None or frame.empty:\n        continue\n    latest_bar_date = frame.index[-1].date()\n    age_days = (today_bkk - latest_bar_date).days\n    if age_days > MAX_STALE_DAILY_BAR_DAYS:\n        stale_short_data[ticker] = f\"Stale daily data: latest bar {latest_bar_date}; {age_days} calendar days old\"\n        continue\n    recent = frame.tail(20)\n    total20_mbaht = float((recent[\"Close\"] * recent[\"Volume\"]).sum() / 1_000_000.0)\n    close = float(recent[\"Close\"].iloc[-1])\n    if not (MIN_PRICE_BAHT <= close <= MAX_PRICE_BAHT and total20_mbaht > MIN_TOTAL20_VALUE_MBAHT):\n        continue\n    stage_rows.append({\"Ticker\": ticker, \"Symbol\": item[\"Symbol\"], \"Market\": item[\"Market\"], \"Sector\": item[\"Sector\"], \"LastPrice\": close, \"Total20TradingValueMBaht\": total20_mbaht})\n    meta_by_ticker[ticker] = {\"Market\": item[\"Market\"], \"Sector\": item[\"Sector\"]}\n\nstage = pd.DataFrame(stage_rows)\nif stage.empty:\n    raise RuntimeError(\"No stocks passed the price and total-20-session traded-value gates; stopping safely.\")\nprint(f\"Price + total 20-session value + fresh-data pass: {len(stage)}\")\nprint(f\"Excluded for stale daily data (> {MAX_STALE_DAILY_BAR_DAYS} days): {len(stale_short_data)}\")\n\n# Stage 2: slower market-cap lookup only for stocks passing the price/liquidity gates.\ncap_values = []\nfor i, ticker in enumerate(stage[\"Ticker\"], start=1):\n    cap_baht = get_market_cap_baht(ticker)\n    cap_mbaht = cap_baht / 1_000_000.0 if np.isfinite(cap_baht) else np.nan\n    cap_values.append(cap_mbaht)\n    if i == 1 or i % 10 == 0 or i == len(stage):\n        print(f\"Market cap {i}/{len(stage)} | {ticker}\")\nstage[\"MarketCapMBaht\"] = cap_values\nstage = stage[np.isfinite(stage[\"MarketCapMBaht\"]) & (stage[\"MarketCapMBaht\"] > MIN_MARKET_CAP_MBAHT)].copy()\nif stage.empty:\n    raise RuntimeError(\"No stocks passed market cap > THB 5,000m; no signals produced.\")\nprint(f\"Full universe gate passed: {len(stage)} stocks\")\n\n# Stage 3: get ~1 year of daily data for indicator warm-up and scan latest completed bar.\nstock_data = {}\nmetadata_by_symbol = {}\nscan_errors = {}\nstale_indicator_data = {}\nfor i, row in enumerate(stage.iterrows(), start=1):\n    _, record = row\n    ticker = record[\"Ticker\"]\n    symbol = record[\"Symbol\"]\n    try:\n        frame = download_yahoo_history(ticker, period=\"1y\", attempts=2)\n        if len(frame) < 80:\n            scan_errors[ticker] = f\"Insufficient completed daily bars: {len(frame)}\"\n            continue\n        latest_bar_date = frame.index[-1].date()\n        age_days = (pd.Timestamp.now(tz=\"Asia/Bangkok\").date() - latest_bar_date).days\n        if age_days > MAX_STALE_DAILY_BAR_DAYS:\n            stale_indicator_data[ticker] = f\"Stale indicator data: latest bar {latest_bar_date}; {age_days} calendar days old\"\n            continue\n        stock_data[ticker] = frame\n        metadata_by_symbol[ticker] = {\n            \"Market\": record[\"Market\"],\n            \"Sector\": record[\"Sector\"],\n            \"MarketCapMBaht\": float(record[\"MarketCapMBaht\"]),\n            \"Total20TradingValueMBaht\": float(record[\"Total20TradingValueMBaht\"]),\n        }\n    except Exception as exc:\n        scan_errors[ticker] = f\"{type(exc).__name__}: {exc}\"\n    if i == 1 or i % 10 == 0 or i == len(stage):\n        print(f\"Indicator history {i}/{len(stage)} | loaded={len(stock_data)} errors={len(scan_errors)}\")\n\nif not stock_data:\n    raise RuntimeError(\"No qualifying stocks have fresh, sufficient OHLCV history. Stopping without output signals.\")\n\n# Enforce one common latest completed trading date across every published row.\nexpected_scan_date = max(frame.index[-1].date() for frame in stock_data.values())\nfor ticker in list(stock_data):\n    ticker_date = stock_data[ticker].index[-1].date()\n    if ticker_date != expected_scan_date:\n        stale_indicator_data[ticker] = (\n            f\"Latest bar {ticker_date} does not match common scan date {expected_scan_date}\"\n        )\n        stock_data.pop(ticker)\n        metadata_by_symbol.pop(ticker, None)\nif not stock_data:\n    raise RuntimeError(\"No stocks share a common latest completed daily bar; stopping without publishing.\")\nofficial_set_latest_date = get_official_set_latest_trading_date(expected_scan_date)\nif expected_scan_date != official_set_latest_date:\n    raise RuntimeError(\n        \"MARKET_DATE_MISMATCH: latest completed Yahoo Finance bar \"\n        f\"{expected_scan_date} does not match SET's official latest trading date \"\n        f\"{official_set_latest_date}. Refusing to publish stale or incomplete scan data.\"\n    )\nprint(\"Official market-date validation: PASS\")\nprint(f\"Common completed scan date: {expected_scan_date}\")\nprint(f\"Excluded for stale/mismatched indicator data: {len(stale_indicator_data)}\")\n\n\nSCAN_MODE = \"AFTER_HOURS_DAILY\"\nif SCAN_MODE == \"INTRADAY_15M\":\n    # Attach free intraday prices as reference fields only; daily signals remain unchanged.\n    from datetime import time as dt_time\n    \n    BANGKOK_TZ = \"Asia/Bangkok\"\n    now_bkk = pd.Timestamp.now(tz=BANGKOK_TZ)\n    clock = now_bkk.time()\n    weekday_session = now_bkk.weekday() < 5\n    inside_set_session = weekday_session and (\n        dt_time(10, 0) <= clock <= dt_time(12, 30)\n        or dt_time(14, 30) <= clock <= dt_time(16, 30)\n    )\n    # A Yahoo 15-minute bar timestamp marks the BAR START, not its close.\n    # Require a completed 15-minute bar no older than 30 minutes; exclude rather than disguise provider lag.\n    INTRADAY_BAR_MINUTES = 15\n    INTRADAY_MAX_AGE_MINUTES = 30\n    INTRADAY_RETRY_ATTEMPTS = 2\n    intraday_quote_by_ticker = {}\n    intraday_data_by_ticker = {}\n    intraday_quote_errors = {}\n    \n    def _intraday_frame_for_ticker(raw, ticker, batch_size):\n        if raw is None or raw.empty:\n            return pd.DataFrame()\n        item = pd.DataFrame()\n        if isinstance(raw.columns, pd.MultiIndex):\n            for level in range(raw.columns.nlevels):\n                if ticker in set(map(str, raw.columns.get_level_values(level))):\n                    try:\n                        item = raw.xs(ticker, axis=1, level=level, drop_level=True).copy()\n                        break\n                    except Exception:\n                        continue\n            if item.empty and batch_size == 1:\n                item = raw.copy()\n        elif batch_size == 1:\n            item = raw.copy()\n        if item.empty:\n            return pd.DataFrame()\n        item.columns = [str(column).strip().title() for column in item.columns]\n        item = item.loc[:, ~item.columns.duplicated()].copy()\n        required = [\"Open\", \"High\", \"Low\", \"Close\", \"Volume\"]\n        if any(column not in item.columns for column in required):\n            return pd.DataFrame()\n        item = item[required].copy()\n        for column in required:\n            item[column] = pd.to_numeric(item[column], errors=\"coerce\")\n        item = item.dropna(subset=required)\n        item = item[item[\"Close\"] > 0].sort_index()\n        if item.empty:\n            return pd.DataFrame()\n        item.index = pd.to_datetime(item.index)\n        if getattr(item.index, \"tz\", None) is None:\n            item.index = item.index.tz_localize(\"UTC\")\n        item.index = item.index.tz_convert(BANGKOK_TZ)\n        return item\n    \n    freshness_tickers = [str(ticker) for ticker in stock_data.keys()]\n    raw_intraday = pd.DataFrame()\n    if freshness_tickers:\n        try:\n            raw_intraday = yf.download(\n                freshness_tickers, period=\"10d\", interval=\"15m\",\n                auto_adjust=False, progress=False, threads=True, group_by=\"ticker\",\n            )\n        except Exception as exc:\n            intraday_quote_errors[\"__BATCH__\"] = f\"{type(exc).__name__}: {exc}\"\n    \n    for ticker in freshness_tickers:\n        try:\n            intraday = _intraday_frame_for_ticker(raw_intraday, ticker, len(freshness_tickers))\n            # Ignore an in-progress 15-minute candle: it is not a completed price observation.\n            if not intraday.empty:\n                intraday = intraday[\n                    intraday.index + pd.Timedelta(minutes=INTRADAY_BAR_MINUTES) <= now_bkk\n                ]\n            # Retry stale/missing batch results as single-symbol requests. This can recover\n            # per-symbol bars omitted or delayed by Yahoo's multi-ticker endpoint.\n            for attempt in range(1, INTRADAY_RETRY_ATTEMPTS + 1):\n                needs_retry = intraday.empty or len(intraday) < max(_scanner_module.CONFIG.ema_long + 3, _scanner_module.CONFIG.bb_window + 3)\n                if not intraday.empty:\n                    candidate_start = intraday.index[-1]\n                    candidate_end = candidate_start + pd.Timedelta(minutes=INTRADAY_BAR_MINUTES)\n                    candidate_age = (now_bkk - candidate_end).total_seconds() / 60.0\n                    needs_retry = not (0 <= candidate_age <= INTRADAY_MAX_AGE_MINUTES)\n                if not needs_retry:\n                    break\n                try:\n                    raw_one = yf.download(\n                        ticker, period=\"10d\", interval=\"15m\", auto_adjust=False,\n                        progress=False, threads=False, timeout=20,\n                    )\n                    refreshed = _intraday_frame_for_ticker(raw_one, ticker, 1)\n                    if not refreshed.empty:\n                        refreshed = refreshed[\n                            refreshed.index + pd.Timedelta(minutes=INTRADAY_BAR_MINUTES) <= now_bkk\n                        ]\n                    if not refreshed.empty and (intraday.empty or len(refreshed) > len(intraday) or refreshed.index[-1] > intraday.index[-1]):\n                        intraday = refreshed\n                except Exception as retry_exc:\n                    print(f\"INTRADAY_RETRY_FAILED {ticker} attempt={attempt}: {type(retry_exc).__name__}\")\n                if attempt < INTRADAY_RETRY_ATTEMPTS and intraday.empty:\n                    time.sleep(1.0 * attempt)\n    \n            if intraday.empty:\n                intraday_quote_errors[ticker] = \"Yahoo Finance returned no intraday Close bars after retry\"\n                continue\n            quote_time = intraday.index[-1]  # provider timestamp = start of the 15-minute bar\n            quote_price = float(intraday[\"Close\"].iloc[-1])\n            bar_close_time = quote_time + pd.Timedelta(minutes=INTRADAY_BAR_MINUTES)\n            age_minutes = (now_bkk - bar_close_time).total_seconds() / 60.0\n            quote_fresh = 0 <= age_minutes <= INTRADAY_MAX_AGE_MINUTES\n            intraday_data_by_ticker[ticker] = intraday.copy()\n            intraday_quote_by_ticker[ticker] = {\n                \"LatestIntradayPrice\": quote_price,\n                \"IntradayBarTimeBangkok\": quote_time.isoformat(),\n                \"IntradayBarCloseTimeBangkok\": bar_close_time.isoformat(),\n                \"IntradayAgeMinutes\": round(age_minutes, 2),\n                \"IntradayPriceFreshnessStatus\": \"FRESH\" if quote_fresh else \"STALE\",\n            }\n            if inside_set_session and not quote_fresh:\n                intraday_quote_errors[ticker] = (\n                    f\"Intraday bar stale/future after retries: bar_start={quote_time.isoformat()}, \"\n                    f\"bar_close={bar_close_time.isoformat()}, age_since_close={age_minutes:.1f} min\"\n                )\n        except Exception as exc:\n            intraday_quote_errors[ticker] = f\"{type(exc).__name__}: {exc}\"\n    \n    print(\"Intraday source: Yahoo Finance free 15m bars; actual exchange delay is not guaranteed\")\n    print(\"Bangkok check time:\", now_bkk.isoformat())\n    print(\"Inside expected SET weekday session:\", inside_set_session)\n    print(\"Intraday bars retrieved:\", len(intraday_quote_by_ticker), \"/\", len(freshness_tickers))\n    print(\"Intraday freshness errors:\", len(intraday_quote_errors))\n    # During market hours, fail closed PER TICKER rather than discarding every valid\n    # signal because one Yahoo symbol is delayed. Only tickers with verified fresh\n    # intraday bars remain eligible for the published scan; exclusions force PARTIAL.\n    INTRADAY_CANDIDATE_COUNT = len(freshness_tickers)\n    intraday_exclusions = {}\n    if inside_set_session:\n        for _ticker in list(stock_data):\n            _quote = intraday_quote_by_ticker.get(_ticker, {})\n            _issue = intraday_quote_errors.get(_ticker)\n            if _issue or _quote.get(\"IntradayPriceFreshnessStatus\") != \"FRESH\":\n                _reason = _issue or \"No verified fresh intraday bar available\"\n                intraday_exclusions[_ticker] = _reason\n                stock_data.pop(_ticker, None)\n                metadata_by_symbol.pop(_ticker, None)\n        print(\"Qualifying tickers before intraday freshness gate:\", INTRADAY_CANDIDATE_COUNT)\n        print(\"Excluded due to stale/missing intraday data:\", len(intraday_exclusions))\n        print(\"Tickers eligible for current-session scan:\", len(stock_data))\n        if not stock_data:\n            raise RuntimeError(\n                \"STALE_DATA: no qualifying ticker had a verified fresh intraday bar during \"\n                \"market hours. No current scan report should be published.\"\n            )\n    \n    results = scan_universe(stock_data, metadata_by_symbol, CONFIG)\n    # Current entry/warning signals are calculated from completed 15-minute OHLCV candles.\n    # The daily close remains the reference for the agreed THB 1-80 universe price filter.\n    INTRADAY_SIGNALS_IMPLEMENTED = callable(scan_intraday_signal)\n    INTRADAY_SIGNAL_CALCULATION_COMPLETE = False\n    intraday_signal_errors = {}\n    if not results.empty:\n        _intraday_signal_rows = []\n        for _, _result_row in results.iterrows():\n            _ticker = str(_result_row[\"Symbol\"]).strip().upper()\n            if not _ticker.endswith(\".BK\"):\n                _ticker += \".BK\"\n            _frame = intraday_data_by_ticker.get(_ticker)\n            if _frame is None or _frame.empty:\n                _intraday_signal_rows.append({\"IntradaySignalError\": \"No completed intraday OHLCV\"})\n                continue\n            try:\n                _intraday_signal_rows.append(scan_intraday_signal(_frame, CONFIG))\n            except Exception as _signal_exc:\n                _intraday_signal_rows.append({\"IntradaySignalError\": f\"{type(_signal_exc).__name__}: {_signal_exc}\"})\n        for _row_pos, _intraday_signal in enumerate(_intraday_signal_rows):\n            if \"IntradaySignalError\" in _intraday_signal:\n                results.loc[_row_pos, \"EARLY_ENTRY\"] = False\n                results.loc[_row_pos, \"EARLY_WARNING\"] = \"\"\n                results.loc[_row_pos, \"ENTRY_SCORE\"] = 0\n                results.loc[_row_pos, \"SignalReasons\"] = \"Intraday signal unavailable: \" + _intraday_signal[\"IntradaySignalError\"]\n                _failed_symbol = str(results.loc[_row_pos, \"Symbol\"])\n                intraday_signal_errors[_failed_symbol] = _intraday_signal[\"IntradaySignalError\"]\n                scan_errors[_failed_symbol] = _intraday_signal[\"IntradaySignalError\"]\n                continue\n            for _field, _value in _intraday_signal.items():\n                results.loc[_row_pos, _field] = _value\n        INTRADAY_SIGNAL_CALCULATION_COMPLETE = bool(_intraday_signal_rows) and all(\n            \"IntradaySignalError\" not in _signal for _signal in _intraday_signal_rows\n        )\n    # Attach verified intraday quote fields for freshness checks and report audit.\n    _intraday_quote_for_symbol = lambda symbol: intraday_quote_by_ticker.get(\n        (str(symbol).strip().upper() if str(symbol).strip().upper().endswith(\".BK\") else str(symbol).strip().upper() + \".BK\"), {}\n    )\n    if not results.empty:\n        results[\"LatestIntradayPrice\"] = results[\"Symbol\"].map(\n            lambda symbol: _intraday_quote_for_symbol(symbol).get(\"LatestIntradayPrice\", np.nan)\n        )\n        results[\"IntradayBarTimeBangkok\"] = results[\"Symbol\"].map(\n            lambda symbol: _intraday_quote_for_symbol(symbol).get(\"IntradayBarTimeBangkok\", \"\")\n        )\n        results[\"IntradayAgeMinutes\"] = results[\"Symbol\"].map(\n            lambda symbol: _intraday_quote_for_symbol(symbol).get(\"IntradayAgeMinutes\", np.nan)\n        )\n        results[\"IntradayPriceFreshnessStatus\"] = results[\"Symbol\"].map(\n            lambda symbol: _intraday_quote_for_symbol(symbol).get(\"IntradayPriceFreshnessStatus\", \"UNAVAILABLE\")\n        )\n        if inside_set_session:\n            intraday_ok = (\n                pd.to_numeric(results[\"LatestIntradayPrice\"], errors=\"coerce\").gt(0)\n                & pd.to_datetime(results[\"IntradayBarTimeBangkok\"], errors=\"coerce\").notna()\n                & pd.to_numeric(results[\"IntradayAgeMinutes\"], errors=\"coerce\").between(0, INTRADAY_MAX_AGE_MINUTES)\n                & results[\"IntradayPriceFreshnessStatus\"].eq(\"FRESH\")\n            )\n            if not intraday_ok.all():\n                bad_symbols = results.loc[~intraday_ok, \"Symbol\"].astype(str).head(10).tolist()\n                raise RuntimeError(\n                    \"STALE_DATA: intraday prices could not be mapped and validated for every result row. \"\n                    f\"Bad symbols: {bad_symbols}. No current scan report should be published.\"\n                )\n    \nelse:\n    # After-hours mode uses completed daily bars only; no intraday data is implied.\n    from datetime import time as dt_time\n    BANGKOK_TZ = \"Asia/Bangkok\"\n    now_bkk = pd.Timestamp.now(tz=BANGKOK_TZ)\n    inside_set_session = False\n    INTRADAY_BAR_MINUTES = 15\n    INTRADAY_MAX_AGE_MINUTES = 30\n    intraday_quote_by_ticker = {}\n    intraday_data_by_ticker = {}\n    intraday_quote_errors = {}\n    intraday_exclusions = {}\n    INTRADAY_CANDIDATE_COUNT = len(stock_data)\n    INTRADAY_SIGNALS_IMPLEMENTED = callable(scan_intraday_signal)\n    INTRADAY_SIGNAL_CALCULATION_COMPLETE = False\n    intraday_signal_errors = {}\n    results = scan_universe(stock_data, metadata_by_symbol, CONFIG)\n    results[\"LatestIntradayPrice\"] = np.nan\n    results[\"IntradayBarTimeBangkok\"] = \"\"\n    results[\"IntradayAgeMinutes\"] = np.nan\n    results[\"IntradayPriceFreshnessStatus\"] = \"NOT_APPLICABLE_DAILY_MODE\"\n    results[\"SignalSuppressedReason\"] = \"\"\n    print(\"SCAN MODE: AFTER_HOURS_DAILY\")\n    print(\"Signals use the latest verified completed daily OHLCV bar, not intraday bars.\")\n    print(\"Intraday freshness: NOT APPLICABLE in after-hours daily mode.\")\n\nif results.empty:\n    raise RuntimeError(\"Scanner returned no rows; stopping without publishing.\")\nif results[\"Date\"].isna().any() or pd.to_datetime(results[\"Date\"], errors=\"coerce\").dt.date.nunique() != 1:\n    raise RuntimeError(\"REPORT_DATA_QUALITY_FAILED: scan rows do not share one valid scan date; refusing to publish.\")\nif (results[\"UniverseStatus\"] == \"SCAN_ERROR\").any():\n    scan_errors[\"SCANNER_INTERNAL\"] = f\"{int((results['UniverseStatus'] == 'SCAN_ERROR').sum())} internal scan errors\"\nresults = results.sort_values([\"EARLY_ENTRY\", \"ENTRY_SCORE\"], ascending=[False, False], na_position=\"last\")\nsignals = results[(results[\"EARLY_ENTRY\"] == True) | (results[\"EARLY_WARNING\"].isin([\"WATCH\", \"WARNING\"]))].copy()\n\nprint(\"\\n\" + \"=\" * 90)\nprint(\"15-MINUTE INTRADAY SIGNALS + FRESHNESS CHECK\")\nprint(\"No backtest | no orders | completed 15m OHLCV drives signals | free Yahoo source only\")\nprint(\"=\" * 90)\nprint(\"Classification source:\", CLASSIFICATION_SOURCE_USED)\nprint(\"Stocks passing all universe filters:\", len(stage))\nprint(\"Stocks with indicator data:\", len(stock_data))\nprint(\"Scan errors:\", len(scan_errors))\nprint(\"Stale/mismatched data exclusions:\", len(stale_short_data) + len(stale_indicator_data))\nprint(\"EARLY ENTRY:\", int((results[\"EARLY_ENTRY\"] == True).sum()))\nprint(\"WATCH/WARNING:\", int(results[\"EARLY_WARNING\"].isin([\"WATCH\", \"WARNING\"]).sum()))\ndisplay(signals if not signals.empty else results.head(20))\n\n# Export the scan report from this Colab session (not to GitHub/main).\noutput_path = \"/content/fast_hybrid_pro_live_scan.csv\"\nresults.to_csv(output_path, index=False, encoding=\"utf-8-sig\")\nprint(\"CSV report saved:\", output_path)\ntry:\n    from google.colab import files\n    files.download(output_path)\nexcept Exception:\n    print(\"Not running in Google Colab; CSV remains at\", output_path)\n\nif scan_errors:\n    print(\"\\nFirst 20 scan errors:\")\n    for ticker, message in list(scan_errors.items())[:20]:\n        print(f\"{ticker}: {message}\")\nif intraday_exclusions:\n    print(\"\\nFirst 20 symbols excluded for stale/missing intraday data:\")\n    for ticker, message in list(intraday_exclusions.items())[:20]:\n        print(f\"{ticker}: {message}\")\n", "<Fast_Hybrid_Pro_After_Hours_Cell>", "exec"), globals(), globals())

In [ ]:
# Publish a durable status for BOTH successful and failed runs.
# Failure diagnostics are redacted and truncated; credentials are not intentionally included.
import base64
import json
import requests
import re
from datetime import datetime, timezone

REPORT_BRANCH = "scan-reports"
REPORT_REPO = "worasak2530seed/fast-hybrid-pro"
REPORT_API = "https://api.github.com/repos/{}/contents".format(REPORT_REPO)
REPORT_HEADERS = None

def _get_report_token(allow_prompt=False):
    token = None
    try:
        from google.colab import userdata
        for secret_name in ("FAST_HYBRID_GITHUB_TOKEN", "GITHUB_TOKEN", "GH_TOKEN"):
            try:
                token = userdata.get(secret_name)
            except Exception:
                token = None
            if token:
                break
    except Exception:
        token = None
    if not token and allow_prompt:
        from getpass import getpass
        token = getpass("GitHub token (input hidden; never paste it into a code cell): ").strip()
    return token

def _publish_report_file(path, raw_bytes, message):
    url = "{}/{}".format(REPORT_API, path)
    existing = requests.get(url, headers=REPORT_HEADERS, params={"ref": REPORT_BRANCH}, timeout=30)
    payload = {"message": message, "content": base64.b64encode(raw_bytes).decode("ascii"), "branch": REPORT_BRANCH}
    if existing.status_code == 200:
        payload["sha"] = existing.json()["sha"]
    elif existing.status_code != 404:
        existing.raise_for_status()
    response = requests.put(url, headers=REPORT_HEADERS, json=payload, timeout=45)
    response.raise_for_status()
    return response.json()["content"]["html_url"]

if globals().get("SCAN_RUN_STATUS") == "FAILED":
    # Keep the previous successful CSV intact; it is not the output of this failed run.
    RUN_ID = str(globals().get("SCAN_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
    UPDATED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
    error_type = str(globals().get("SCAN_RUN_ERROR_TYPE") or "UnknownError")[:200]
    error_message = str(globals().get("SCAN_RUN_ERROR_MESSAGE") or "Scanner failed without an exception message")[:4000]
    traceback_text = str(globals().get("SCAN_RUN_TRACEBACK") or "No traceback was captured.")[-16000:]
    traceback_text = re.sub(r"(?i)Bearer\s+\S+", "Bearer [REDACTED]", traceback_text)
    error_message = re.sub(r"(?i)Bearer\s+\S+", "Bearer [REDACTED]", error_message)
    token = _get_report_token(allow_prompt=False)
    if not token:
        print("GITHUB_FAILURE_REPORT_BLOCKED: Colab secret FAST_HYBRID_GITHUB_TOKEN/GITHUB_TOKEN is unavailable.")
        print("Redacted error details are printed above, but could not be uploaded automatically.")
        raise RuntimeError("GITHUB_FAILURE_REPORT_BLOCKED: add the GitHub Contents read/write token to Colab Secrets.")
    REPORT_HEADERS = {
        "Authorization": "Bearer " + token,
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "Fast-Hybrid-Pro-Scanner",
    }
    branch_check = requests.get("https://api.github.com/repos/{}/branches/{}".format(REPORT_REPO, REPORT_BRANCH), headers=REPORT_HEADERS, timeout=30)
    branch_check.raise_for_status()
    trace_path = "reports/latest_scan_failure.txt"
    trace_url = _publish_report_file(trace_path, traceback_text.encode("utf-8"), "Publish scanner failure traceback " + RUN_ID)
    failure_summary = {
        "run_id": RUN_ID, "status": "FAILED", "updated_at_utc": UPDATED_AT,
        "scan_date": None, "error_type": error_type, "error_message": error_message,
        "traceback_path": trace_path,
        "traceback_url": "https://github.com/{}/blob/{}/{}".format(REPORT_REPO, REPORT_BRANCH, trace_path),
        "status_url": "https://github.com/{}/{}/blob/{}/reports/latest_scan_status.json".format(*REPORT_REPO.split("/"), REPORT_BRANCH),
        "note": "This run failed. The previous successful CSV is intentionally preserved and must not be treated as the result of this failed run."
    }
    summary_path = "reports/latest_scan_summary.json"
    summary_url = _publish_report_file(summary_path, json.dumps(failure_summary, ensure_ascii=False, indent=2).encode("utf-8"), "Publish scanner failure summary " + RUN_ID)
    failure_status = dict(failure_summary)
    failure_status["summary_path"] = summary_path
    failure_status["summary_url"] = "https://github.com/{}/blob/{}/{}/{}".format(REPORT_REPO, REPORT_BRANCH, summary_path.split("/")[0], summary_path.split("/")[1])
    failure_status["report_url"] = None
    status_url = _publish_report_file("reports/latest_scan_status.json", json.dumps(failure_status, ensure_ascii=False, indent=2).encode("utf-8"), "Publish scanner FAILED status " + RUN_ID)
    print("GITHUB_FAILURE_REPORT_PUBLISH_PASS")
    print("PUBLISH_STATUS: FAILED")
    print("RUN_ID:", RUN_ID)
    print("ERROR_TYPE:", error_type)
    print("ERROR_MESSAGE:", error_message)
    print("SUMMARY_REPORT:", summary_url)
    print("TRACEBACK_REPORT:", trace_url)
    print("STATUS_REPORT:", status_url)
else:
    # Publish this completed scan to the dedicated scan-reports branch.
    # Requires a Colab secret named GITHUB_TOKEN with Contents: Read and write access
    # to worasak2530seed/fast-hybrid-pro. No token is printed or saved in the notebook.
    import base64
    import json
    from datetime import datetime, timezone
    
    # Fail clearly if the scan cell above was skipped or failed; never publish partial state.
    if "results" not in globals() or not isinstance(globals().get("results"), pd.DataFrame):
        raise RuntimeError("SCAN_NOT_RUN: run the complete scan cell immediately above and fix any error before publishing.")
    if "signals" not in globals() or not isinstance(globals().get("signals"), pd.DataFrame):
        raise RuntimeError("SCAN_NOT_RUN: scan output 'signals' is missing; rerun the scan cell before publishing.")
    
    # Fail closed on malformed data before publishing any report.
    if results.empty:
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: empty scan result; refusing to publish.")
    if "Symbol" not in results or results["Symbol"].astype(str).str.strip().eq("").any():
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: missing or blank symbols; refusing to publish.")
    if results["Symbol"].astype(str).str.upper().duplicated().any():
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: duplicate symbols; refusing to publish.")
    if "Date" not in results:
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: missing Date column; refusing to publish.")
    report_dates = pd.to_datetime(results["Date"], errors="coerce")
    if report_dates.isna().any() or report_dates.dt.date.nunique() != 1:
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: invalid or mixed scan dates; refusing to publish.")
    if "UniversePass" not in results or not results["UniversePass"].fillna(False).astype(bool).all():
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: non-qualifying stock present; refusing to publish.")
    if "ENTRY_SCORE" not in results or not pd.to_numeric(results["ENTRY_SCORE"], errors="coerce").between(0, 5).all():
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: ENTRY_SCORE outside expected 0..5 range; refusing to publish.")
    if "Close" not in results or not pd.to_numeric(results["Close"], errors="coerce").gt(0).all():
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: invalid Close values; refusing to publish.")
    if "EARLY_ENTRY" not in results or "EARLY_WARNING" not in results:
        raise RuntimeError("REPORT_DATA_QUALITY_FAILED: required signal columns missing; refusing to publish.")
    if bool(globals().get("inside_set_session", False)):
        required_intraday = [
            "LatestIntradayPrice", "IntradayBarTimeBangkok",
            "IntradayAgeMinutes", "IntradayPriceFreshnessStatus",
        ]
        if any(column not in results.columns for column in required_intraday):
            raise RuntimeError("REPORT_DATA_QUALITY_FAILED: intraday freshness fields missing during market hours; refusing to publish.")
        intraday_ok = (
            pd.to_numeric(results["LatestIntradayPrice"], errors="coerce").gt(0)
            & pd.to_datetime(results["IntradayBarTimeBangkok"], errors="coerce").notna()
            & pd.to_numeric(results["IntradayAgeMinutes"], errors="coerce").between(
                0, float(globals().get("INTRADAY_MAX_AGE_MINUTES", 30))
            )
            & results["IntradayPriceFreshnessStatus"].eq("FRESH")
        )
        if not intraday_ok.all():
            raise RuntimeError("REPORT_DATA_QUALITY_FAILED: one or more result rows failed intraday freshness validation; refusing to publish.")
    print("REPORT_DATA_QUALITY_CHECK_PASS")
    INTEGRITY_CHECKS = [
        "non-empty report",
        "non-blank unique symbols",
        "single valid scan date",
        "all rows pass universe gate",
        "ENTRY_SCORE within 0..5",
        "positive Close",
        "required signal columns present",
        "intraday verification is reported separately and only passes during the expected market session when every row is fresh",
    ]
    
    
    REPORT_BRANCH = "scan-reports"
    REPORT_REPO = "worasak2530seed/fast-hybrid-pro"
    REPORT_API = f"https://api.github.com/repos/{REPORT_REPO}/contents"
    RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    UPDATED_AT = datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z")
    SCAN_DATE = str(pd.to_datetime(results["Date"], errors="coerce").max().date()) if "Date" in results and results["Date"].notna().any() else "unknown"
    
    GITHUB_TOKEN = None
    try:
        from google.colab import userdata
        for secret_name in ("FAST_HYBRID_GITHUB_TOKEN", "GITHUB_TOKEN", "GH_TOKEN"):
            try:
                GITHUB_TOKEN = userdata.get(secret_name)
            except Exception:
                GITHUB_TOKEN = None
            if GITHUB_TOKEN:
                break
    except Exception:
        GITHUB_TOKEN = None
    
    if not GITHUB_TOKEN:
        # Fallback for Colab sessions where Secrets are unavailable/not authorized.
        # The token is entered interactively, hidden by getpass, and is not written to the notebook.
        from getpass import getpass
        GITHUB_TOKEN = getpass("GitHub token (input hidden; never paste it into a code cell): ").strip()
    
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_REPORT_PUBLISH_FAILED: no GitHub token was provided. "
            "Add FAST_HYBRID_GITHUB_TOKEN (preferred) or GITHUB_TOKEN in Colab Secrets, or rerun this cell and enter the token at the hidden prompt. "
            "The local CSV was already created; no Telegram success notification should be sent."
        )
    
    REPORT_HEADERS = {
        "Authorization": f"Bearer {GITHUB_TOKEN}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
        "User-Agent": "Fast-Hybrid-Pro-Scanner",
    }
    
    # Verify authentication and branch access before attempting any report writes.
    auth_check = requests.get(
        f"https://api.github.com/repos/{REPORT_REPO}/branches/{REPORT_BRANCH}",
        headers=REPORT_HEADERS,
        timeout=30,
    )
    if auth_check.status_code in (401, 403):
        raise RuntimeError(
            "GITHUB_REPORT_PUBLISH_FAILED: GitHub rejected the token or it lacks repository access. "
            "Confirm the token belongs to the correct GitHub account and has Contents: Read and write "
            "for worasak2530seed/fast-hybrid-pro. Do not paste the token into chat."
        )
    if auth_check.status_code == 404:
        raise RuntimeError(
            "GITHUB_REPORT_PUBLISH_FAILED: branch scan-reports was not found or is not visible to this token."
        )
    auth_check.raise_for_status()
    print("GITHUB_TOKEN_AUTH_CHECK_PASS")
    
    def publish_report_file(path, raw_bytes, message):
        url = f"{REPORT_API}/{path}"
        existing = requests.get(url, headers=REPORT_HEADERS, params={"ref": REPORT_BRANCH}, timeout=30)
        payload = {
            "message": message,
            "content": base64.b64encode(raw_bytes).decode("ascii"),
            "branch": REPORT_BRANCH,
        }
        if existing.status_code == 200:
            payload["sha"] = existing.json()["sha"]
        elif existing.status_code != 404:
            existing.raise_for_status()
        response = requests.put(url, headers=REPORT_HEADERS, json=payload, timeout=45)
        response.raise_for_status()
        return response.json()["content"]["html_url"]
    
    # Fail closed for stale bars even outside market hours. Preserve raw labels for
    # diagnostics, but suppress all actionable labels unless each row has a fresh bar.
    _raw_entry = results["EARLY_ENTRY"].fillna(False).astype(bool).copy()
    _raw_warning = results["EARLY_WARNING"].fillna("").astype(str).copy()
    _freshness_columns = [
        "LatestIntradayPrice", "IntradayBarTimeBangkok",
        "IntradayAgeMinutes", "IntradayPriceFreshnessStatus",
    ]
    if str(globals().get("SCAN_MODE", "")) == "AFTER_HOURS_DAILY":
        # Daily closed-bar signals are valid for an after-hours watchlist, not live execution.
        _fresh_mask = pd.Series(True, index=results.index)
    else:
        _fresh_mask = pd.Series(False, index=results.index)
        if all(column in results.columns for column in _freshness_columns):
            _fresh_mask = (
                pd.to_numeric(results["LatestIntradayPrice"], errors="coerce").gt(0)
                & pd.to_datetime(results["IntradayBarTimeBangkok"], errors="coerce").notna()
                & pd.to_numeric(results["IntradayAgeMinutes"], errors="coerce").between(
                    0, float(globals().get("INTRADAY_MAX_AGE_MINUTES", 30))
                )
                & results["IntradayPriceFreshnessStatus"].astype(str).eq("FRESH")
            )
    results["RawEARLY_ENTRY"] = _raw_entry
    results["RawEARLY_WARNING"] = _raw_warning
    results["SignalSuppressedReason"] = ""
    stale_rows = ~_fresh_mask
    results.loc[stale_rows, "SignalSuppressedReason"] = "STALE_OR_UNVERIFIED_INTRADAY_DATA"
    results.loc[stale_rows, "EARLY_ENTRY"] = False
    results.loc[stale_rows, "EARLY_WARNING"] = ""
    # Rebuild the signal list from the publishable labels only; stale candidates stay
    # visible in the audit CSV but cannot appear in the actionable signal summary.
    signals = results.loc[
        results["EARLY_ENTRY"].fillna(False).astype(bool)
        | results["EARLY_WARNING"].fillna("").astype(str).isin(["WATCH", "WARNING"])
    ].copy()
    raw_early_entry_count = int(_raw_entry.sum())
    raw_watch_warning_count = int(_raw_warning.isin(["WATCH", "WARNING"]).sum())
    stale_intraday_row_count = int(stale_rows.sum())
    all_published_intraday_fresh = bool(not results.empty and _fresh_mask.all())
    # Upload data first; publish SUCCESS status last so Termux cannot announce success early.
    csv_bytes = results.to_csv(index=False).encode("utf-8-sig")
    summary_rows = []
    for _, row in signals.iterrows():
        summary_rows.append({
            "Symbol": str(row.get("Symbol", "")),
            "Date": str(row.get("Date", "")),
            "Close": None if pd.isna(row.get("Close")) else float(row.get("Close")),
            "SignalPrice": None if pd.isna(row.get("SignalPrice")) else float(row.get("SignalPrice")),
            "SignalTime": str(row.get("SignalTime", "")),
            "EARLY_ENTRY": bool(row.get("EARLY_ENTRY", False)),
            "ENTRY_SCORE": None if pd.isna(row.get("ENTRY_SCORE")) else int(row.get("ENTRY_SCORE")),
            "EARLY_WARNING": str(row.get("EARLY_WARNING", "")),
            "SignalReasons": str(row.get("SignalReasons", "")),
            "ReferenceTarget7Pct": None if pd.isna(row.get("ReferenceTarget7Pct")) else float(row.get("ReferenceTarget7Pct")),
            "ReferenceTarget15Pct": None if pd.isna(row.get("ReferenceTarget15Pct")) else float(row.get("ReferenceTarget15Pct")),
        })
    # All stale rows are already suppressed in the published CSV.
    stale_intraday_symbols = results.loc[stale_rows, "Symbol"].astype(str).tolist() if "Symbol" in results.columns else []
    _after_hours_daily = str(globals().get("SCAN_MODE", "")) == "AFTER_HOURS_DAILY"
    publish_status = (
        "SUCCESS"
        if (
            _after_hours_daily and not scan_errors and not stale_short_data
            and not stale_indicator_data and not results.empty
            and pd.to_datetime(results["Date"], errors="coerce").dt.date.nunique() == 1
        ) or (
            not _after_hours_daily and bool(inside_set_session) and not scan_errors
            and not intraday_exclusions and stale_intraday_row_count == 0
            and all_published_intraday_fresh
            and bool(globals().get("INTRADAY_SIGNALS_IMPLEMENTED", False))
            and bool(globals().get("INTRADAY_SIGNAL_CALCULATION_COMPLETE", False))
        )
        else "PARTIAL"
    )

    summary = {
        "integrity_check_passed": True,
        "integrity_checks": INTEGRITY_CHECKS,
        "run_id": RUN_ID,
        "status": publish_status,
        "scan_date": SCAN_DATE,
        "updated_at_utc": UPDATED_AT,
        "universe_pass_count": int(len(stage)),
        "indicator_data_count": int(len(stock_data)),
        "scan_error_count": int(len(scan_errors)),
        "scan_errors_detail": {str(k): str(v) for k, v in scan_errors.items()},
        "intraday_signal_calculation_complete": bool(globals().get("INTRADAY_SIGNAL_CALCULATION_COMPLETE", False)),
        "intraday_signal_error_count": int(len(globals().get("intraday_signal_errors", {}))),
        "intraday_signal_errors": {str(k): str(v) for k, v in globals().get("intraday_signal_errors", {}).items()},
        "stale_data_excluded_count": int(len(stale_short_data) + len(stale_indicator_data) + len(intraday_exclusions)),
        "stale_intraday_row_count": stale_intraday_row_count,
        "stale_intraday_signals_suppressed": True,
        "stale_intraday_symbols": stale_intraday_symbols,
        "signals_eligible_as_verified_live": bool(publish_status == "SUCCESS" and not _after_hours_daily),
        "signal_use_status": ("AFTER_HOURS_DAILY_CLOSED_BAR" if _after_hours_daily and publish_status == "SUCCESS" else ("VERIFIED_CURRENT_SESSION" if publish_status == "SUCCESS" else "UNVERIFIED_DO_NOT_TREAT_AS_LIVE")),
        "stale_data_exclusions": {**stale_short_data, **stale_indicator_data, **intraday_exclusions},
        "early_entry_count": int((results["EARLY_ENTRY"] == True).sum()),
        "raw_early_entry_candidate_count_before_freshness_suppression": raw_early_entry_count,
        "watch_warning_count": int(results["EARLY_WARNING"].isin(["WATCH", "WARNING"]).sum()),
        "raw_watch_warning_candidate_count_before_freshness_suppression": raw_watch_warning_count,
        "signal_count": int(len(signals)),
        "signals": summary_rows,
        "report_path": "reports/latest_scan.csv",
        "summary_path": "reports/latest_scan_summary.json",
        "data_note": "20-session traded value is estimated as Close × Volume; it is not the exchange's exact traded-value field.",
        "intraday_source_note": "Yahoo Finance free 15-minute interval bars; availability and actual exchange delay are not guaranteed. Completed 15-minute OHLCV drives entry/warning signals; daily close is used for universe price eligibility.",
        "scan_mode": "AFTER_HOURS_DAILY_CLOSED_BAR" if _after_hours_daily else "INTRADAY_15M_SIGNALS",
        "signal_calculation_basis": "completed 15-minute OHLCV",
        "intraday_signals_implemented": bool(globals().get("INTRADAY_SIGNALS_IMPLEMENTED", False)),
        "intraday_signal_calculation_complete": bool(globals().get("INTRADAY_SIGNAL_CALCULATION_COMPLETE", False)),
        "intraday_freshness_check": {
            "inside_expected_set_weekday_session": bool(inside_set_session),
            "max_bar_age_minutes": int(INTRADAY_MAX_AGE_MINUTES),
            "stale_intraday_signals_suppressed": True,
            "retrieved_ticker_count": int(len(intraday_quote_by_ticker)),
            "qualifying_ticker_count": int(INTRADAY_CANDIDATE_COUNT),
            "published_ticker_count": int(len(stock_data)),
            "excluded_stale_ticker_count": int(len(intraday_exclusions)),
            "stale_intraday_row_count_in_published_results": stale_intraday_row_count,
            "stale_intraday_symbols_in_published_results": stale_intraday_symbols,
            "error_count": int(len(intraday_quote_errors)),
            "excluded_symbols": intraday_exclusions,
            "verified_for_market_session": bool(
                inside_set_session
                and bool(globals().get("INTRADAY_SIGNAL_CALCULATION_COMPLETE", False))
                and not scan_errors
                and not intraday_exclusions
                and all_published_intraday_fresh
            ),
        },
    }
    summary_bytes = json.dumps(summary, ensure_ascii=False, indent=2).encode("utf-8")
    status = {
        "integrity_check_passed": True,
        "integrity_checks": INTEGRITY_CHECKS,
        "run_id": RUN_ID,
        "status": publish_status,
        "detail": (
            "After-hours daily-bar scan completed; signals are watchlist candidates, not live intraday execution." if _after_hours_daily else "Verified 15-minute intraday signal scan completed and reports published."
            if publish_status == "SUCCESS" else
            f"PARTIAL: {len(scan_errors)} scan error(s), {len(intraday_exclusions)} stale/missing intraday exclusion(s), {stale_intraday_row_count} published row(s) with stale/missing intraday bars; inside_set_session={bool(inside_set_session)}. Signals are NOT verified live and must not be treated as current-session signals. A verified SUCCESS requires completed 15-minute OHLCV signals and fresh bars for every published row."
        ),
        "scan_date": SCAN_DATE,
        "updated_at_utc": UPDATED_AT,
        "report_path": "reports/latest_scan.csv",
        "summary_path": "reports/latest_scan_summary.json",
        "summary_url": f"https://github.com/{REPORT_REPO}/blob/{REPORT_BRANCH}/reports/latest_scan_summary.json",
        "report_url": f"https://github.com/{REPORT_REPO}/blob/{REPORT_BRANCH}/reports/latest_scan.csv",
    }
    status_bytes = json.dumps(status, ensure_ascii=False, indent=2).encode("utf-8")
    
    csv_url = publish_report_file("reports/latest_scan.csv", csv_bytes, f"Publish scanner CSV report {RUN_ID}")
    summary_url = publish_report_file("reports/latest_scan_summary.json", summary_bytes, f"Publish scanner summary {RUN_ID}")
    # Replace any traceback left by an older run with a diagnostic record for THIS run.
    # This prevents a stale failure file from being mistaken for the current result.
    diagnostics = {
        "run_id": RUN_ID,
        "status": publish_status,
        "updated_at_utc": UPDATED_AT,
        "scan_error_count": int(len(scan_errors)),
        "scan_errors_detail": {str(k): str(v) for k, v in scan_errors.items()},
        "intraday_signal_error_count": int(len(globals().get("intraday_signal_errors", {}))),
        "intraday_signal_errors": {str(k): str(v) for k, v in globals().get("intraday_signal_errors", {}).items()},
        "stale_data_excluded_count": int(len(stale_short_data) + len(stale_indicator_data) + len(intraday_exclusions)),
        "stale_intraday_row_count": stale_intraday_row_count,
        "stale_intraday_symbols": stale_intraday_symbols,
        "signals_eligible_as_verified_live": bool(publish_status == "SUCCESS"),
        "signal_use_status": "VERIFIED_CURRENT_SESSION" if publish_status == "SUCCESS" else "UNVERIFIED_DO_NOT_TREAT_AS_LIVE",
        "stale_data_exclusions": {**stale_short_data, **stale_indicator_data, **intraday_exclusions},
        "note": "This diagnostic file belongs to the latest published run; it replaces any traceback from an older run."
    }
    diagnostics_url = publish_report_file("reports/latest_scan_failure.txt", json.dumps(diagnostics, ensure_ascii=False, indent=2).encode("utf-8"), f"Publish latest scanner diagnostics {RUN_ID}")
    status_url = publish_report_file("reports/latest_scan_status.json", status_bytes, f"Publish scanner status {RUN_ID}")
    print("GITHUB_REPORT_PUBLISH_PASS")
    print("PUBLISH_STATUS:", publish_status)
    print("RUN_ID:", RUN_ID)
    print("SCAN_DATE:", SCAN_DATE)
    print("EARLY_ENTRY:", summary["early_entry_count"])
    print("WATCH_WARNING:", summary["watch_warning_count"])
    print("SCAN_ERRORS:", summary["scan_error_count"])
    print("STALE_DATA_EXCLUDED:", summary["stale_data_excluded_count"])
    print("CSV_REPORT:", csv_url)
    print("SUMMARY_REPORT:", summary_url)
    print("STATUS_REPORT:", status_url)
    
    
